# Vanguard 13F — factor risk, straight from the Atoti cube in Python

This notebook reproduces every view in the **Vanguard 13F filings** folder by calling the Atoti cube
**directly from Python** — `cube.query(...)` in the kernel, no FastAPI, no HTTP, no Streamlit.
Each view below is a self-contained, hand-written query feeding a pandas table (grids) or an
altair chart (charts). The view JSON files were the *spec*; the code here is the literal API.

Run order: build the cube once (next cell), then any view cell. Built on free/public data
(SEC 13F + EDGAR XBRL + Stooq), Vanguard Group's 13F book as a weight overlay.

In [1]:
# The notebook container is air-gapped and its filesystem is read-only, so altair/narwhals
# are staged on the host at data/_pylibs (visible read-only as /app/data/_pylibs).
# No-op outside the container, where altair is installed in the barra/ venv normally.
import os, sys
_libs = "/app/data/_pylibs"
if os.path.isdir(_libs) and _libs not in sys.path:
    sys.path.insert(0, _libs)

import datetime as dt
import numpy as np
import pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()   # whole-market book: sector/day frames exceed altair's 5k default
import notebook_helpers as N

session, cube = N.build()                       # builds the six-frame cube on :9096 (~1-2 min)
h, l, m = cube.hierarchies, cube.levels, cube.measures
D = dt.date(2026, 6, 30)                         # latest monthly COB in the sample; all views as-of D
PREC = 3                                         # decimals shown in every grid (the views' `prec`; 3.576%)
MANAGER = l["Manager"] == "Vanguard"
print("cube ready — hierarchies:", sorted(n for _, n in h))

cube built: 6,004,074 leaf rows, 22 style factors, 7 scenario sets, 123 PIT sets, 68,425 scenario-day rows


cube load: 19.9s on :9097
cube ready — hierarchies: ['Book', 'CorrStress', 'Date', 'Day', 'DayDate', 'DaySet', 'FactorDim', 'Manager', 'PITSet', 'PositionRank', 'ScenarioDay', 'ScenarioSet', 'Security', 'StressShock']


## Direct cube access — the two idioms

Everything below is built from two `cube.query` shapes: **(1)** scalar measures grouped by a
level under a `filter`, and **(2)** the **`Day` / `DayDate` levels** of the day-facts table
(`ScenarioDays`), which hold a scenario P&L *vector* as one fact per day — so a vector measure
(`PnL at day`) becomes an ordinary group-by, sliced by **`DaySet`** (its own set hierarchy, same
member names as `ScenarioSet`). The synthetic `ScenarioDay` parameter hierarchy that older runs
of this notebook used is kept in the cube for backward compatibility only (~10-40× slower, and
`× Sector` fails on the largest books).

In [2]:
# (1) scalar measures, grouped by ScenarioSet, sliced to Vanguard @ latest COB — "the switch":
#     the same measures, every scenario mode, side by side.
display(cube.query(
    m["Scenario VaR 99"], m["Scenario worst loss"], m["Total VaR 99"],
    mode="raw", levels=[l["ScenarioSet"]], filter=MANAGER & (l["Date"] == D)))

# (2) the Day/DayDate levels give the COVID P&L vector as a per-day series (head). NB the day
#     measures read the DaySet hierarchy, not ScenarioSet -- slice DaySet to the set you want.
display(cube.query(
    m["PnL at day"],
    mode="raw", levels=[l["Day"], l["DayDate"]],
    filter=MANAGER & (l["Date"] == D) & (l["DaySet"] == "Evt:COVID2020")).head())

,ScenarioSet,Scenario VaR 99,Scenario worst loss,Total VaR 99
0,Evt:COVID2020,0.107628,0.133986,0.107725
1,Evt:Rates2022,0.040909,0.042677,0.041162
2,Evt:Selloff2018,0.035754,0.036759,0.036043
3,HistFull,0.035078,0.133986,0.035372
4,Hypo:MomentumCrash,0.002392,0.002392,0.005145
5,Hypo:RiskOff,0.00276,0.00276,0.005326
6,Hypo:ValueRotation,0.002653,0.002653,0.005271


,Day,DayDate,PnL at day
0,0,2020-02-03,0.008371
1,1,2020-02-04,0.021373
2,2,2020-02-05,0.012848
3,3,2020-02-06,0.003698
4,4,2020-02-07,-0.010212


## L1 · Book risk summary

In [3]:
# ── L1 · Book risk summary ────────────────────────────────────────────────────────────────
# The top of the drill-down: one row, the whole book. Vanguard's 1-day 99% VaR right now, split into
# the factor-driven tail (Scenario VaR 99, historical simulation) and the idiosyncratic tail
# (Specific vol), then combined (Total VaR 99). Everything below decomposes THIS number.
# Direct API: no group-by beyond Book; slice to Vanguard / latest COB / the full-history set.
df = cube.query(
    m["Total VaR 99"], m["Scenario VaR 99"], m["Specific vol"],
    mode="raw", levels=[l["Manager"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
)
N.style_grid(df, prec=PREC)

,Book,Total VaR 99,Scenario VaR 99,Specific vol
0,Vanguard,3.537%,3.508%,0.196%


In [ ]:
# ── L1 · Book risk summary, in dollars ───────────────────────────────────────────────────
# The same cell priced in money — via the cube's Units CONTEXT, not a second measure name
# (2026-08-22 refactor: the old "<measure> $" twins are gone). `Book MV` is the 13F market value
# of the sliced book at D (the SEC's $000→$ unit change of 2023 is normalised in the builder, so
# the history reads in one unit) and is always dollars. Every OTHER weight-unit measure toggles
# weight <-> dollars on ONE switch: slice l["Units"] == "$" and the SAME measure names (Total VaR
# 99, Scenario VaR 99, Specific vol) read at Book MV scale — the Pivot lens's `units: $` control
# is this same switch. Whole dollars in the grid (PREC does not apply to money columns); Book MV
# ≈ $6.4tn for Vanguard at 2026-06-30.
df = cube.query(
    m["Book MV"], m["Total VaR 99"], m["Scenario VaR 99"], m["Specific vol"],
    mode="raw", levels=[l["Manager"]],
    filter=BOOK & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull") & (l["Units"] == "$"),
)
N.style_grid(df, prec=PREC, money=["Total VaR 99", "Scenario VaR 99", "Specific vol"])


## L2 · Factor contributions

In [4]:
# ── L2 · Factor contributions ─────────────────────────────────────────────────────────────
# Drill the book VaR onto the risk factors. Marginal Scenario VaR 99 is each factor's P&L on the
# book's 1%-tail scenario (ADDITIVE — the factors sum to the book factor-VaR); % of Scenario VaR
# 99 is its share. "Which factors own the tail?" — here Market dominates a long-equity book.
# Direct API: group by Factor; same Vanguard / COB / HistFull slice; sorted biggest-first.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df, prec=PREC)

,FactorGroup,Factor,Marginal Scenario VaR 99,% of Scenario VaR 99
11,Market,Market,2.891%,82.277%
7,Industry,Ind:Information Technology,0.322%,9.177%
17,Style,NdxBeta,0.286%,8.135%
12,Style,Beta,0.285%,8.113%
19,Style,RateBeta,0.154%,4.395%
6,Industry,Ind:Industrials,0.056%,1.608%
5,Industry,Ind:Health Care,0.052%,1.470%
22,Style,Value,0.035%,0.984%
3,Industry,Ind:Energy,0.033%,0.935%
0,Industry,Ind:Communication Services,0.027%,0.758%


## L2 · Factor incremental VaR

In [5]:
# ── L2 · Factor incremental VaR ───────────────────────────────────────────────────────────
# The diversification-aware companion to contributions: Incremental Scenario VaR 99 is how much
# book VaR is RELEASED if a factor's exposure is removed (the book tail recomputed without it).
# Unlike the marginals it is NOT additive (VaR is sub-additive) — "what does cutting this factor
# actually buy me?". Direct API: group by Factor; sort by the incremental column.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["Incremental Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Incremental Scenario VaR 99", ascending=False)
N.style_grid(df, prec=PREC)

,FactorGroup,Factor,Marginal Scenario VaR 99,Incremental Scenario VaR 99
11,Market,Market,2.891%,2.546%
12,Style,Beta,0.285%,0.285%
7,Industry,Ind:Information Technology,0.322%,0.092%
17,Style,NdxBeta,0.286%,0.057%
19,Style,RateBeta,0.154%,0.043%
5,Industry,Ind:Health Care,0.052%,0.043%
0,Industry,Ind:Communication Services,0.027%,0.014%
6,Industry,Ind:Industrials,0.056%,0.011%
4,Industry,Ind:Financials,0.010%,0.001%
8,Industry,Ind:Materials,0.000%,0.000%


## L2 · Issuer contributions

In [6]:
# ── L2 · Issuer contributions ─────────────────────────────────────────────────────────────
# Same decomposition, now by name. Marginal Total VaR 99 is each issuer's additive share of the
# COMBINED tail (factor + specific in quadrature — the Euler split), meaningful per-name where
# idiosyncratic risk lives. "Which positions carry the book's risk?". Direct API: group by Issuer
# (the cube returns the Country/Sector/Issuer security path); top names first.
df = cube.query(
    m["Marginal Total VaR 99"], m["% of Total VaR 99"],
    mode="raw", levels=[l["Issuer"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Total VaR 99", ascending=False)
N.style_grid(df, prec=PREC)

,Country,Sector,Issuer,Marginal Total VaR 99,% of Total VaR 99
2939,US,Information Technology,NVIDIA CORP,0.368%,10.395%
2913,US,Information Technology,MICROSOFT CORP,0.207%,5.846%
2714,US,Information Technology,Apple Inc.,0.202%,5.692%
2708,US,Information Technology,Alphabet Inc.,0.168%,4.738%
2747,US,Information Technology,Broadcom Inc.,0.161%,4.538%
2921,US,Information Technology,"Meta Platforms, Inc.",0.113%,3.193%
301,US,Consumer Discretionary,AMAZON COM INC,0.099%,2.798%
601,US,Consumer Discretionary,"Tesla, Inc.",0.080%,2.249%
2973,US,Information Technology,Palantir Technologies Inc.,0.040%,1.141%
2953,US,Information Technology,ORACLE CORP,0.038%,1.065%


## L2 · Issuer incremental VaR

In [7]:
# ── L2 · Issuer incremental VaR ───────────────────────────────────────────────────────────
# Per-name diversification view: Incremental Total VaR 99 = book Total VaR released by removing
# the name (factor tail re-struck on the reduced book + its specific variance stripped). The cut
# list a PM reads to de-risk. Direct API: group by Issuer; sort by the incremental column.
df = cube.query(
    m["Marginal Total VaR 99"], m["Incremental Total VaR 99"],
    mode="raw", levels=[l["Issuer"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Incremental Total VaR 99", ascending=False)
N.style_grid(df, prec=PREC)

,Country,Sector,Issuer,Marginal Total VaR 99,Incremental Total VaR 99
2939,US,Information Technology,NVIDIA CORP,0.368%,0.301%
2913,US,Information Technology,MICROSOFT CORP,0.207%,0.204%
2714,US,Information Technology,Apple Inc.,0.202%,0.192%
2708,US,Information Technology,Alphabet Inc.,0.168%,0.167%
2747,US,Information Technology,Broadcom Inc.,0.161%,0.129%
2921,US,Information Technology,"Meta Platforms, Inc.",0.113%,0.105%
301,US,Consumer Discretionary,AMAZON COM INC,0.099%,0.096%
601,US,Consumer Discretionary,"Tesla, Inc.",0.080%,0.064%
2973,US,Information Technology,Palantir Technologies Inc.,0.040%,0.040%
2953,US,Information Technology,ORACLE CORP,0.038%,0.038%


## L3 · FactorGroup contributions

In [8]:
# ── L3 · FactorGroup contributions ────────────────────────────────────────────────────────
# Roll the factor contributions up one level — Market vs Style — for the one-line "is this a
# market bet or a style bet?" read. Same additive Marginal Scenario VaR 99 / share, grouped at
# the FactorGroup level of the FactorDim hierarchy. Direct API: levels=[FactorGroup].
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["FactorGroup"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df, prec=PREC)

,FactorGroup,Marginal Scenario VaR 99,% of Scenario VaR 99
1,Market,2.891%,82.277%
2,Style,0.380%,10.808%
0,Industry,0.243%,6.915%


## L3 · Sector contributions

In [9]:
# ── L3 · Sector contributions ─────────────────────────────────────────────────────────────
# The book's risk by GICS sector — additive Marginal Total VaR 99 per sector (factor + specific),
# the cross-sectional concentration view. Group at the Sector level of the Security hierarchy (the
# cube returns Country/Sector). Direct API: levels=[Sector]; biggest contributors first.
df = cube.query(
    m["Marginal Total VaR 99"], m["% of Total VaR 99"],
    mode="raw", levels=[l["Sector"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Total VaR 99", ascending=False)
N.style_grid(df, prec=PREC)

,Country,Sector,Marginal Total VaR 99,% of Total VaR 99
62,US,Information Technology,1.928%,54.428%
61,US,Industrials,0.507%,14.297%
59,US,Financials,0.313%,8.834%
56,US,Consumer Discretionary,0.308%,8.692%
60,US,Health Care,0.213%,6.007%
55,US,Communication Services,0.080%,2.269%
63,US,Materials,0.072%,2.027%
58,US,Energy,0.070%,1.963%
13,Canada,Financials,0.014%,0.399%
17,Canada,Materials,0.014%,0.397%


## Stress · Factor contributions (COVID 2020)

In [10]:
# ── Stress · Factor contributions (COVID 2020) ────────────────────────────────────────────
# The L2 factor decomposition re-struck under a STRESS set instead of the full history: the ONLY
# change from "L2 · Factor contributions" is the ScenarioSet slice (Evt:COVID2020). That one
# switch — slicing the ScenarioSet hierarchy — turns historical-sim VaR into event-replay stress
# VaR, and Market's tail balloons. Direct API: identical query, ScenarioSet == 'Evt:COVID2020'.
df = cube.query(
    m["Marginal Scenario VaR 99"], m["% of Scenario VaR 99"],
    mode="raw", levels=[l["Factor"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020"),
).sort_values("Marginal Scenario VaR 99", ascending=False)
N.style_grid(df, prec=PREC)

,FactorGroup,Factor,Marginal Scenario VaR 99,% of Scenario VaR 99
11,Market,Market,12.742%,95.099%
12,Style,Beta,0.659%,4.918%
21,Style,Size,0.400%,2.983%
9,Industry,Ind:Real Estate,0.241%,1.796%
1,Industry,Ind:Consumer Discretionary,0.222%,1.656%
18,Style,NonLinSize,0.204%,1.525%
16,Style,Momentum,0.089%,0.666%
4,Industry,Ind:Financials,0.073%,0.546%
10,Industry,Ind:Utilities,0.046%,0.346%
13,Style,EarnYield,0.018%,0.131%


## VaR trend (HistFull)

In [11]:
# ── VaR trend (HistFull) ──────────────────────────────────────────────────────────────────
# The book's risk THROUGH TIME: 99% factor VaR, total VaR, and specific vol at every monthly COB
# (the full 2016-2024 calendar, historical-sim set). One line per measure — "is the book getting
# riskier?". Direct API: group by Date (no Date filter -> the whole series); melt the three
# measure columns to long form for a colour-per-measure line in altair.
trend = cube.query(
    m["Scenario VaR 99"], m["Total VaR 99"], m["Specific vol"],
    mode="raw", levels=[l["Date"]],
    filter=MANAGER & (l["ScenarioSet"] == "HistFull"),
).reset_index().sort_values("Date")
meas = ["Scenario VaR 99", "Total VaR 99", "Specific vol"]
trend["Date"] = pd.to_datetime(trend["Date"])
trend[meas] = trend[meas].astype(float)
long = trend.melt(id_vars="Date", value_vars=meas, var_name="Measure", value_name="Value")

alt.Chart(long).mark_line(point=True).encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("Value:Q", axis=alt.Axis(format="%"), title="% of NAV"),
    color=alt.Color("Measure:N", scale=alt.Scale(domain=meas,
        range=["#2a4d69", "#c46d4e", "#6b8f71"]), legend=alt.Legend(orient="top", title=None)),
    tooltip=["Date:T", "Measure:N", alt.Tooltip("Value:Q", format=".4f")],
).properties(height=380, width="container", title="VaR trend — Vanguard (HistFull)")

alt.Chart(...)

## Stress board — all scenarios

In [12]:
# ── Stress board · all scenarios ──────────────────────────────────────────────────────────
# Every scenario set side by side at the latest COB: factor VaR, worst single-day loss, and total
# VaR per set — historical-sim vs the three event replays vs the hypotheticals. The CRO's "how bad
# across regimes" board. Direct API: group by ScenarioSet (no ScenarioSet filter -> all sets);
# melt to long for grouped horizontal bars, worst-first.
board = cube.query(
    m["Scenario VaR 99"], m["Scenario worst loss"], m["Total VaR 99"],
    mode="raw", levels=[l["ScenarioSet"]],
    filter=MANAGER & (l["Date"] == D),
).reset_index()
meas = ["Scenario VaR 99", "Scenario worst loss", "Total VaR 99"]
board[meas] = board[meas].astype(float)
long = board.melt(id_vars="ScenarioSet", value_vars=meas, var_name="Measure", value_name="Value")

alt.Chart(long).mark_bar().encode(
    y=alt.Y("ScenarioSet:N", title=None, sort="-x"),
    x=alt.X("Value:Q", axis=alt.Axis(format="%"), title="% of NAV"),
    yOffset=alt.YOffset("Measure:N"),
    color=alt.Color("Measure:N", scale=alt.Scale(domain=meas,
        range=["#2a4d69", "#c46d4e", "#6b8f71"]), legend=alt.Legend(orient="top", title=None)),
    tooltip=["ScenarioSet:N", "Measure:N", alt.Tooltip("Value:Q", format=".4f")],
).properties(height=420, width="container", title="Stress board — Vanguard @ 2024-12-31")

alt.Chart(...)

## Tail risk — VaR ladder & Expected Shortfall

The full tail across every scenario set: VaR at 95 / 97.5 / 99, then **Expected Shortfall** (ES /
CVaR) at 97.5 / 99 — the *mean* loss beyond VaR and the Basel FRTB measure that replaced VaR.

In [13]:
# ── Tail risk · VaR ladder & Expected Shortfall ────────────────────────────────
# The full tail, every scenario set side by side: VaR at 95 / 97.5 / 99 (the ladder reads how fat
# the tail is), then Expected Shortfall (ES, a.k.a. CVaR) at 97.5 / 99 — the MEAN loss beyond VaR,
# the Basel FRTB measure that replaced VaR (ES97.5 is the regulatory point). Scenario PnL vol is the
# per-observation dispersion (null for the length-1 hypotheticals). ES >= VaR at the same level by
# construction; ES97.5 ~ VaR99 under a normal tail. Direct API: group by ScenarioSet, all sets.
ladder = cube.query(
    m["Scenario VaR 95"], m["Scenario VaR 97.5"], m["Scenario VaR 99"],
    m["Scenario ES 97.5"], m["Scenario ES 99"], m["Scenario PnL vol"],
    mode="raw", levels=[l["ScenarioSet"]],
    filter=MANAGER & (l["Date"] == D),
).sort_values("Scenario ES 97.5", ascending=False)
N.style_grid(ladder, prec=PREC)

,ScenarioSet,Scenario VaR 95,Scenario VaR 97.5,Scenario VaR 99,Scenario ES 97.5,Scenario ES 99,Scenario PnL vol
0,Evt:COVID2020,5.177%,8.943%,10.763%,10.853%,13.399%,3.830%
1,Evt:Rates2022,3.105%,3.631%,4.091%,3.988%,4.215%,1.685%
3,HistFull,1.819%,2.494%,3.508%,3.815%,5.076%,1.255%
2,Evt:Selloff2018,2.796%,3.489%,3.575%,3.595%,3.676%,1.648%
5,Hypo:RiskOff,0.276%,0.276%,0.276%,0.276%,0.276%,—
6,Hypo:ValueRotation,0.265%,0.265%,0.265%,0.265%,0.265%,—
4,Hypo:MomentumCrash,0.239%,0.239%,0.239%,0.239%,0.239%,—


## L2 · Factor ES contributions

The ES analogue of the factor VaR decomposition: each factor's mean P&L over the book's worst-k
tail scenarios. Additive — the factors sum to the book **Scenario ES 97.5**.

In [14]:
# ── L2 · Factor ES contributions ─────────────────────────────────────
# The ES analogue of "L2 · Factor contributions": each factor's MEAN P&L over the book's worst-k
# tail scenarios (k = 2.5% of the history). Additive — the factors sum to the book Scenario ES 97.5
# — and % of Scenario ES 97.5 is the share. Because ES averages the whole tail (not the single 1%
# day) it is the steadier "which factors own the tail?" read. Direct API: group by Factor, HistFull.
df = cube.query(
    m["Marginal Scenario ES 97.5"], m["% of Scenario ES 97.5"],
    mode="raw", levels=[l["Factor"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Marginal Scenario ES 97.5", ascending=False)
N.style_grid(df, prec=PREC)

,FactorGroup,Factor,Marginal Scenario ES 97.5,% of Scenario ES 97.5
11,Market,Market,3.505%,91.890%
12,Style,Beta,0.310%,8.128%
18,Style,NonLinSize,0.024%,0.634%
7,Industry,Ind:Information Technology,0.013%,0.340%
6,Industry,Ind:Industrials,0.013%,0.339%
19,Style,RateBeta,0.009%,0.229%
21,Style,Size,0.007%,0.182%
3,Industry,Ind:Energy,0.006%,0.170%
17,Style,NdxBeta,0.006%,0.169%
9,Industry,Ind:Real Estate,0.006%,0.162%


## Concentration — Risk HHI

Single-number name concentration: the Herfindahl index of each name's share of book Total VaR.
1/N for an evenly-diversified book up to 1.0 for a single name — the limit-monitoring gauge.

In [15]:
# ── Concentration · Risk HHI ─────────────────────────────────────────
# Single-number name concentration: the Herfindahl index of each name's share of book Total VaR,
# the sum of share^2 over names. 1/N for an evenly-diversified book (here ~0.03 under the full
# history, ~30-odd effective names) up to 1.0 for a single name; the single-factor hypotheticals
# read far higher because the shock concentrates risk in a handful of loaded names. as_pct off —
# HHI is an index, not a percent. Direct API: group by ScenarioSet.
# NB (124-book build, 2026-08-14): on a whole-market book this size (~3.6k names) the
# all-sets Risk HHI query materializes per-name risk shares for every scenario set at
# once and exhausts the notebook cube heap — sliced to HistFull, same as the /trends
# date-loop precedent. (Risk HHI is legacy on the desk anyway — Top-5 risk share
# replaced it in the limits/monitor views.)
hhi = cube.query(
    m["Risk HHI"],
    mode="raw", levels=[l["ScenarioSet"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"),
).sort_values("Risk HHI", ascending=False)
N.style_grid(hhi, pct=False, prec=PREC)

,ScenarioSet,Risk HHI
0,HistFull,0.025


## Scenario P&L — COVID 2020

Two structurally-**different** graphs, built the explicit way: **two `cube.query` calls → two
DataFrames → two graphs**, one graph per DataFrame. The query (a tabular dataset) is the primitive;
the graph just references it. The scenario engine produces a P&L *vector* over the COVID replay
window; the day-facts table holds it one fact per day, so the `Day` / `DayDate` levels read it as
an ordinary group-by. **Query 1** is the book path (`levels=[Day, DayDate]`); **Query 2** adds a
grain (`levels=[Day, DayDate, Sector]`) and is re-ordered worst→best into a loss curve — the
ordering computed **in the DataFrame**, not the chart.

> Migration note (2026-08-15): these cells were rewritten from the legacy `ScenarioDay`
> parameter-hierarchy idiom onto the `Day`/`DayDate` levels + `PnL at day` (the fast day-facts
> path). The code below is the new idiom; the stored outputs are from the last executed run of the
> old one — the numbers are identical day for day (pinned by
> `test_risk_measures.py::t_day_path_ties_scenario_day_and_foots_by_sector`), only the query is
> different. Re-run to refresh the outputs.

In [16]:
# ── COVID 2020 · QUERY 1 of 2 -> the book P&L path DataFrame ───────────────────────────────
# Two different graphs need two different queries -> two DataFrames. This is query 1: the per-day
# MANAGER path. The ScenarioDays table holds the P&L vector one fact per day, so the path is a plain
# group-by: levels=[Day, DayDate] (DayDate is the calendar date as a LEVEL, read off the axis).
# `PnL at day` reads the DaySet hierarchy; the 99% VaR threshold and the worst-loss day ride along
# as book-level marker measures lifted over the day hierarchies (they read ScenarioSet), so the
# filter names the set on BOTH hierarchies. Epoch-day int (worst date) -> real date in pandas.
covid = (MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "Evt:COVID2020")
         & (l["DaySet"] == "Evt:COVID2020"))
covid_path = cube.query(
    m["PnL at day"], m["VaR line at day"], m["Worst pnl at day"], m["Worst date at day (epoch)"],
    mode="raw", levels=[l["Day"], l["DayDate"]], filter=covid,
).reset_index()
for c in ["PnL at day", "VaR line at day", "Worst pnl at day"]:
    covid_path[c] = covid_path[c].astype(float)
covid_path["DayDate"]    = pd.to_datetime(covid_path["DayDate"])   # altair serialises EVERY
covid_path["date"]       = covid_path["DayDate"]                   # column: date -> Timestamp
covid_path["worst_date"] = pd.to_datetime(covid_path["Worst date at day (epoch)"].astype("int64"), unit="D")
covid_path        # DataFrame 1 — the tabular dataset graph 1 draws from

,index,Day,DayDate,PnL at day,VaR line at day,Worst pnl at day,Worst date at day (epoch),date,worst_date
0,0,0,2020-02-03,0.008371,-0.107628,-0.133986,18337,2020-02-03,2020-03-16
1,1,1,2020-02-04,0.021373,-0.107628,-0.133986,18337,2020-02-04,2020-03-16
2,2,2,2020-02-05,0.012848,-0.107628,-0.133986,18337,2020-02-05,2020-03-16
3,3,3,2020-02-06,0.003698,-0.107628,-0.133986,18337,2020-02-06,2020-03-16
4,4,4,2020-02-07,-0.010212,-0.107628,-0.133986,18337,2020-02-07,2020-03-16
...,...,...,...,...,...,...,...,...,...
77,77,77,2020-05-22,0.001500,-0.107628,-0.133986,18337,2020-05-22,2020-03-16
78,78,78,2020-05-26,0.019045,-0.107628,-0.133986,18337,2020-05-26,2020-03-16
79,79,79,2020-05-27,0.020841,-0.107628,-0.133986,18337,2020-05-27,2020-03-16
80,80,80,2020-05-28,-0.008457,-0.107628,-0.133986,18337,2020-05-28,2020-03-16


In [17]:
# ── COVID 2020 · GRAPH 1 — book scenario P&L path (draws from `covid_path`) ────────────────
# The graph just references DataFrame 1: a P&L line + zero baseline + the red 99% VaR rule and the
# red worst-loss point (book-level markers, collapsed to one mark each with aggregate:min).
base     = alt.Chart(covid_path)
zero     = base.mark_rule(color="#d8d5cd").encode(y=alt.datum(0))
line     = base.mark_line(point=True, color="#2a4d69", strokeWidth=1.3).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("PnL at day:Q", axis=alt.Axis(format="%"), title="scenario P&L"),
    tooltip=[alt.Tooltip("date:T"),
             alt.Tooltip("PnL at day:Q", format=".2%", title="P&L")])
var_rule = base.mark_rule(color="#c0392b", strokeDash=[4, 4]).encode(
    y=alt.Y("min(VaR line at day):Q"))                               # one book VaR-99 rule
worst    = base.mark_point(color="#c0392b", size=80, filled=True).encode(
    x=alt.X("min(worst_date):T"), y=alt.Y("min(Worst pnl at day):Q"),
    tooltip=[alt.Tooltip("min(worst_date):T", title="worst date"),
             alt.Tooltip("min(Worst pnl at day):Q", format=".2%", title="worst P&L")])

(zero + line + var_rule + worst).properties(
    height=300, width="container", title="COVID 2020 — book scenario P&L path")

alt.LayerChart(...)

In [18]:
# ── COVID 2020 · QUERY 2 of 2 -> the by-Sector loss-curve DataFrame ────────────────────────
# Query 2: the SAME per-day P&L at a finer grain -- levels=[Day, DayDate, Sector] -- so each day
# splits into its sector contributions (the shape the legacy ScenarioDay path could not serve on the
# largest books). The worst->best ordering is computed HERE, in the DataFrame: roll the sectors back
# up to a book total per day, sort those day-totals ascending (worst/most-negative first), carry the
# `rank` onto every sector row, then physically sort the frame. (Graph 2 then uses sort=None to
# honour this order.) Tick labels stay dates (%d %b); the order is the rank.
covid_sector = cube.query(
    m["PnL at day"],
    mode="raw", levels=[l["Day"], l["DayDate"], l["Sector"]], filter=covid,
).reset_index()
covid_sector["PnL at day"] = covid_sector["PnL at day"].astype(float)
covid_sector["DayDate"] = pd.to_datetime(covid_sector["DayDate"])  # as above: altair cannot
covid_sector["date"] = covid_sector["DayDate"]                     # JSON-encode datetime.date

order = (covid_sector.groupby("Day", as_index=False)["PnL at day"]
                     .sum().sort_values("PnL at day"))                # worst (most negative) first
order["rank"] = range(len(order))
covid_sector = (covid_sector.merge(order[["Day", "rank"]], on="Day")
                            .sort_values(["rank", "Sector"]))         # df now in worst->best order
covid_sector["label"] = covid_sector["date"].dt.strftime("%d %b")
covid_sector      # DataFrame 2 — the tabular dataset graph 2 draws from

,index,Day,DayDate,Country,Sector,PnL at day,date,rank,label
2206,2206,29,2020-03-16,Brazil,Communication Services,-2.718648e-07,2020-03-16,0,16 Mar
2214,2214,29,2020-03-16,Canada,Communication Services,-5.118899e-06,2020-03-16,0,16 Mar
2225,2225,29,2020-03-16,Cayman Islands,Communication Services,-6.675632e-06,2020-03-16,0,16 Mar
2242,2242,29,2020-03-16,Indonesia,Communication Services,-2.774489e-08,2020-03-16,0,16 Mar
2250,2250,29,2020-03-16,Mexico,Communication Services,-4.095128e-08,2020-03-16,0,16 Mar
...,...,...,...,...,...,...,...,...,...
2669,2669,35,2020-03-24,Brazil,Utilities,1.218186e-06,2020-03-24,81,24 Mar
2680,2680,35,2020-03-24,Canada,Utilities,1.644913e-04,2020-03-24,81,24 Mar
2692,2692,35,2020-03-24,Chile,Utilities,4.723502e-08,2020-03-24,81,24 Mar
2704,2704,35,2020-03-24,"Korea, Republic of",Utilities,3.776957e-07,2020-03-24,81,24 Mar


In [19]:
# ── COVID 2020 · GRAPH 2 — scenario P&L by Sector, the loss curve (draws from `covid_sector`) ─
# The graph just references DataFrame 2: sectors stacked to the book total, x in the DataFrame's own
# worst->best row order (sort=None), labelled by date.
alt.Chart(covid_sector).mark_area(opacity=0.85, line={"strokeWidth": 0.4}).encode(
    x=alt.X("label:N", sort=None, title="scenario date (worst → best)",   # sort=None => keep df order
            axis=alt.Axis(labelAngle=-45, labelOverlap=True)),
    y=alt.Y("PnL at day:Q", stack="zero", axis=alt.Axis(format="%"), title="scenario P&L"),
    color=alt.Color("Sector:N", scale=alt.Scale(scheme="set2"),
                    legend=alt.Legend(orient="top", title=None)),
    tooltip=[alt.Tooltip("date:T", title="date"), "Sector:N",
             alt.Tooltip("PnL at day:Q", format=".2%", title="P&L")],
).properties(height=300, width="container", title="COVID 2020 — scenario P&L by Sector")

alt.Chart(...)

## Model vs Price — the bridge (docs/price-var-plan.md)

A second historical-sim engine on RAW STOCK RETURNS, no factor model at all —
`PriceSet` is its own switch hierarchy, mirroring `ScenarioSet` (HistFull + every Evt:*
window). T0/T1 (factor-only / + Gaussian specific) and T4 (Price VaR 99, all priced
names) are cube-native; T2 (the Gaussian specific block replaced by REALIZED daily
residuals) and T3 (today's fixed loadings replaced by each day's own — the model's own
identity r_t = L_i(t)·f_t + u_i,t) are numpy on the frames, the SAME population/calendar
the cube's HistFull vector uses. The four sequential differences sum to T4−T0 exactly.

In [ ]:
# ── Model vs Price · headline pair + T0/T1/T4 from the cube ────────────────────────────────
# T0/T1 read ScenarioSet (factor-only, then + Gaussian specific); T4 + the book's own priced-
# weight-share on its OWN Price-VaR tail day read PriceSet (its own switch hierarchy) — both
# sliced to HistFull. Direct API: no group-by, one cell each.
bridge_scn = cube.query(
    m["Scenario VaR 99"], m["Total VaR 99"],
    mode="raw", filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull"))
bridge_price = cube.query(
    m["Price VaR 99"], m["Price coverage at tail"],
    mode="raw", filter=MANAGER & (l["Date"] == D) & (l["PriceSet"] == "HistFull"))
t0 = float(bridge_scn["Scenario VaR 99"].iloc[0])
t1 = float(bridge_scn["Total VaR 99"].iloc[0])
t4 = float(bridge_price["Price VaR 99"].iloc[0])
cov_tail = float(bridge_price["Price coverage at tail"].iloc[0])
print(f"Total VaR 99 (model)      {t1:.{PREC}%}")
print(f"Price VaR 99 (raw tape)   {t4:.{PREC}%}")
print(f"coverage at price tail    {cov_tail:.1%}")

In [ ]:
# ── Model vs Price · T2/T3 from the frames, the full bridge table ──────────────────────────
# T2: today's loadings, Gaussian specific block replaced by REALIZED daily residuals
# (specific_returns). T3: each day's OWN loadings — i.e. the raw stock return itself, on
# today's covered names (the model's identity r_t = L_i(t)·f_t + u_i,t makes this exact).
# Same population (today's covered/held names) and calendar T0/T1 use.
from barra_factor_risk_cube import load_frames
frames = load_frames()
exp_d = frames["exposures"][frames["exposures"]["Date"] == pd.Timestamp(D)]
L = exp_d.pivot_table(index="Position", columns="Factor", values="Loading", aggfunc="first").fillna(0.0)
wide_fr = frames["factor_returns"].pivot(index="Date", columns="Factor", values="Return").dropna(how="any")
factors = [c for c in L.columns if c in wide_fr.columns]
L, wide_fr = L[factors], wide_fr[factors]
pos = frames["positions"]
asof = pos[(pos["Manager"] == "Vanguard") & (pos["Date"] <= pd.Timestamp(D))]
held = asof[asof["Date"] == asof["Date"].max()].set_index("Position")["Weight"]
w = pd.Series(0.0, index=L.index)
w.loc[w.index.intersection(held.index)] = held.reindex(w.index.intersection(held.index))
x = L.to_numpy().T @ w.to_numpy()
fac_pnl = wide_fr.to_numpy() @ x

u_wide = frames["specific_returns"].pivot_table(index="Date", columns="Position", values="SpecificReturn", aggfunc="last")
spec_pnl = u_wide.reindex(index=wide_fr.index, columns=L.index).fillna(0.0).to_numpy() @ w.to_numpy()
t2 = float(-pd.Series(fac_pnl + spec_pnl).quantile(0.01, interpolation="lower"))

r_wide = frames["stock_returns"].pivot_table(index="Date", columns="Position", values="Return", aggfunc="last")
pnl_t3 = r_wide.reindex(index=wide_fr.index, columns=L.index).fillna(0.0).to_numpy() @ w.to_numpy()
t3 = float(-pd.Series(pnl_t3).quantile(0.01, interpolation="lower"))

bridge = pd.DataFrame({
    "step": ["T0", "T1", "T2", "T3", "T4"],
    "measure": ["Scenario VaR 99", "Total VaR 99", "full-sim (realized residuals)",
                "Price VaR (covered names)", "Price VaR 99"],
    "value": [t0, t1, t2, t3, t4],
})
bridge["term"] = bridge["value"].diff()
N.style_grid(bridge.set_index("step"), prec=PREC)

In [ ]:
# ── Model vs Price · top disagreements (Marginal Total VaR vs Marginal Price VaR) ──────────
# Both marginals are additive Euler tail-day contributions — comparable per name.
# Largest |gap| first.
dis = cube.query(
    m["Marginal Total VaR 99"], m["Marginal Price VaR 99"],
    mode="raw", levels=[l["Position"]],
    filter=MANAGER & (l["Date"] == D) & (l["ScenarioSet"] == "HistFull") & (l["PriceSet"] == "HistFull"),
).reset_index()
tick = dict(zip(frames["securities"]["Position"], frames["securities"]["Ticker"]))
dis["Ticker"] = dis["Position"].map(tick).str.upper()
dis["gap"] = dis["Marginal Price VaR 99"].fillna(0) - dis["Marginal Total VaR 99"].fillna(0)
top = dis.reindex(dis["gap"].abs().sort_values(ascending=False).index).head(10)
N.style_grid(top[["Ticker", "Marginal Total VaR 99", "Marginal Price VaR 99", "gap"]].set_index("Ticker"), prec=PREC)

In [ ]:
# ── Model vs Price · the bridge waterfall (the lens's hero chart) ──────────────────────────
# Five levels, four labelled steps — the deltas are SEQUENTIAL differences, so they sum to
# T4 − T0 exactly (VaR is not additive; sequential differences are). Bars, not floating
# blocks: the level is the number that matters, the step label carries the delta.
steps = pd.DataFrame({
    "step": ["T0 Scenario VaR", "T1 Total VaR", "T2 full-sim", "T3 covered-names", "T4 Price VaR"],
    "VaR": [t0, t1, t2, t3, t4],
})
steps["delta"] = [""] + [f"{d:+.3%}" for d in steps["VaR"].diff().dropna()]
base = alt.Chart(steps).encode(x=alt.X("step:N", sort=None, title=None))
(base.mark_bar(size=52, opacity=0.55) .encode(y=alt.Y("VaR:Q", axis=alt.Axis(format="%"), title="1-day 99% VaR"))
 + base.mark_text(dy=-18, fontSize=11).encode(y="VaR:Q", text=alt.Text("VaR:Q", format=".3%"))
 + base.mark_text(dy=-34, fontSize=10, color="#6b6b63").encode(y="VaR:Q", text="delta:N")
).properties(width=640, height=260, title="Model → Price, one assumption at a time")


In [ ]:
# ── Model vs Price · per-name marginals (the lens's scatter) ───────────────────────────────
# Marginal Total VaR (model) vs Marginal Price VaR (raw tape), both Euler tail-day
# contributions summing to their book totals — like-for-like per name. Dot size = 13F weight;
# the diagonal is agreement. Off-diagonal names are where the two engines tell different
# stories (cross-check the disagreements table above).
sc = dis.copy()
sc["Weight"] = sc["Position"].map(held).fillna(0.0)
pts = alt.Chart(sc).mark_circle(opacity=0.6).encode(
    x=alt.X("Marginal Total VaR 99:Q", axis=alt.Axis(format="%"), title="Marginal Total VaR 99 (model)"),
    y=alt.Y("Marginal Price VaR 99:Q", axis=alt.Axis(format="%"), title="Marginal Price VaR 99 (price)"),
    size=alt.Size("Weight:Q", legend=None),
    tooltip=["Ticker", alt.Tooltip("Marginal Total VaR 99", format=".3%"),
             alt.Tooltip("Marginal Price VaR 99", format=".3%"), alt.Tooltip("Weight", format=".2%")],
)
lim = float(pd.concat([sc["Marginal Total VaR 99"], sc["Marginal Price VaR 99"]]).abs().max()) * 1.05
diag = alt.Chart(pd.DataFrame({"v": [-lim, lim]})).mark_line(strokeDash=[3, 3], color="#6b6b63").encode(x="v:Q", y="v:Q")
(pts + diag).properties(width=420, height=420, title="Where the two engines disagree, per name")


In [ ]:
# ── Model vs Price · priced-weight coverage by day (the lens's sparkline) ──────────────────
# The zero-fill disclosure: on each history day, the share of TODAY'S book weight with a real
# return (a missing return contributes zero P&L). The cube's own coverage vector + its dates
# dual, unpacked — the /backtest vector idiom. Low-coverage early years mean the price sim's
# oldest days under-represent the book; the tail-day figure printed above is the one that
# matters for the VaR itself.
cov_vec = cube.query(m["Price coverage vector"], mode="raw",
                     filter=MANAGER & (l["Date"] == D) & (l["PriceSet"] == "HistFull")).iloc[0, 0]
cov_dates = cube.query(m["Price dates (epoch)"], mode="raw",
                       filter=(l["PriceSet"] == "HistFull")).iloc[0, 0]
cov = pd.DataFrame({"Day": pd.to_datetime(np.asarray(cov_dates), unit="s"),
                    "Priced share": np.asarray(cov_vec, dtype=float)})
alt.Chart(cov).mark_line().encode(
    x=alt.X("Day:T", title=None),
    y=alt.Y("Priced share:Q", axis=alt.Axis(format="%"), scale=alt.Scale(domain=[0, 1])),
).properties(width=640, height=120, title="Share of today's book priced, day by day")
